# 03 Small Files and File Layout

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Diagnose a table's physical layout (file count, file sizes, clustering of the filter columns) with one reusable report, understand where small files come from, and fix them at the source (write-time sizing, optimized writes) and afterwards (<code>OPTIMIZE</code>, liquid clustering, predictive optimization).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"A job creates thousands of small files. What now?" is one of the scenarios this learning path asks you to rehearse. Layout problems make every downstream read slow and are invisible in the code: the query looks fine, but it opens 50,000 tiny files. A layout report turns that into numbers you can act on.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Healthy layout</b>
<ul><li><b>File size</b>: tens of MB to about 1 GB per file. Databricks targets ~1 GB for <code>OPTIMIZE</code>, smaller for frequently updated tables</li><li><b>File count</b>: proportional to data size. Thousands of files for a few GB is a red flag</li><li><b>Clustering</b>: files have narrow min/max ranges for the columns you filter on, so data skipping works</li></ul>
Where small files come from:<br>| Source | Why | Fix |<br>|---|---|---|<br>| Frequent small appends (micro-batches, streaming) | Each write adds files | Auto compaction, optimized writes, scheduled <code>OPTIMIZE</code> |<br>| Too many partitions at write time | One file per task per output folder | <code>coalesce</code>/<code>repartition</code> before writing, optimized writes |<br>| Over-partitioning (<code>partitionBy</code> high cardinality) | One folder per value, tiny files in each | Coarser partitions, or liquid clustering |<br>| <code>MERGE</code> / <code>UPDATE</code> with copy-on-write | Rewritten files | Deletion vectors, <code>OPTIMIZE</code> |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A streaming job wrote IoT events every 30 seconds into a table partitioned by <code>device_id</code> (8,000 devices). After a month the table had 9 million files averaging 40 KB, and a one-day query took 20 minutes just planning. The fix: remove the partitioning, cluster by <code>(event_date, device_id)</code> with liquid clustering, enable optimized writes and auto compaction, and run a one-off <code>OPTIMIZE</code>. The same query took 15 seconds.
</div>

## Setup: a table with a bad layout

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a table the way a careless micro-batch job would: 60 small appends with many partitions each.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Several hundred small files.
</div>

In [0]:
DB = "workspace.optimization"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
import time
from pyspark.sql import functions as F

table = f"{DB}.readings_layout"
spark.sql(f"DROP TABLE IF EXISTS {table}")
spark.sql(f"CREATE TABLE {table} (device_id INT, reading DOUBLE, event_ts TIMESTAMP)")
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'false', 'delta.autoOptimize.autoCompact' = 'false')")

for batch in range(60):
    (spark.range(2000).repartition(8)                                  # 8 tasks -> 8 files per batch
     .select((F.rand(seed=batch) * 500).cast("int").alias("device_id"),
             (F.rand(seed=batch + 100) * 100).alias("reading"),
             F.expr(f"timestampadd(MINUTE, {batch}, timestamp'2024-11-01 00:00:00')").alias("event_ts"))
     .write.mode("append").saveAsTable(table))
print("written 60 micro-batches")

## 1. A reusable layout report

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines <code>layout_report(table, column)</code>: file count, total and average file size, the share of small files, and how many files a typical point filter on <code>column</code> would need to read (from per-file min/max).<br><br>
<b>Why it matters</b><br>One function gives you the numbers to decide whether a table needs compaction or clustering, and to prove the improvement afterwards.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Around 480 files with an average size of a few KB, most of them "small", and every file overlapping a typical <code>device_id</code> value.
</div>

In [0]:
def layout_report(table, column):
    d = spark.sql(f"DESCRIBE DETAIL {table}").first()
    files = (spark.table(table).groupBy(F.col("_metadata.file_path").alias("f"))
             .agg(F.min(column).alias("lo"), F.max(column).alias("hi"), F.max("_metadata.file_size").alias("size")))
    stats = files.agg(F.count("*").alias("n"), F.avg("size").alias("avg"),
                      F.sum(F.when(F.col("size") < 8 * 1024 * 1024, 1).otherwise(0)).alias("small")).first()
    probe = files.agg(F.expr(f"percentile_approx((lo + hi) / 2, 0.5)")).first()[0]
    overlapping = files.filter((F.col("lo") <= probe) & (F.col("hi") >= probe)).count()
    print(f"{table.split('.')[-1]}: files={d['numFiles']}  size={d['sizeInBytes'] / 1024 / 1024:.1f} MB  "
          f"avg file={stats['avg'] / 1024:.1f} KB  small files={stats['small'] / stats['n']:.0%}  "
          f"files a point filter on {column} must read={overlapping}/{stats['n']}")

layout_report(table, "device_id")

## 2. The cost: planning and reading many tiny files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Times a selective query on the bad layout.<br><br>
<b>Why it matters</b><br>Each file adds listing, metadata and open costs. With hundreds of thousands of files, this overhead dominates even simple queries.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A timing to compare with section 4.
</div>

In [0]:
def timed_query(label):
    start = time.time()
    n = spark.table(table).filter("device_id = 123").agg(F.avg("reading")).collect()
    print(f"{label:<28} {time.time() - start:.2f}s")

timed_query("before (small files)")

## 3. Fix at the source: fewer, larger files per write

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows two write-time fixes on a copy of one micro-batch: <code>coalesce</code> before writing, and Databricks <b>optimized writes</b>.<br><br>
<b>Why it matters</b><br>The cheapest small file is the one never written. Reducing the number of write tasks, or letting optimized writes shuffle data into fewer files, keeps tables healthy from day one.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same batch written as 8 files without a fix, and as 1 file with <code>coalesce(1)</code>.
</div>

In [0]:
batch = spark.range(2000).repartition(8).select(F.col("id").cast("int").alias("device_id"), F.rand().alias("reading"), F.current_timestamp().alias("event_ts"))
probe_t = f"{DB}.write_probe"
for label, df in [("8 write tasks", batch), ("coalesce(1)", batch.coalesce(1))]:
    df.write.mode("overwrite").saveAsTable(probe_t)
    print(f"{label:<14} -> files: {spark.sql(f'DESCRIBE DETAIL {probe_t}').first()['numFiles']}")
spark.sql(f"DROP TABLE {probe_t}")

# On Databricks: let the platform size files at write time
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true')")

## 4. Fix afterwards: liquid clustering + OPTIMIZE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Enables liquid clustering on <code>device_id</code>, runs <code>OPTIMIZE</code>, and re-runs the layout report and the query.<br><br>
<b>Why it matters</b><br><code>OPTIMIZE</code> compacts the existing small files, and liquid clustering makes it also co-locate rows by <code>device_id</code>, so point filters read only a few files. After this, auto compaction and predictive optimization keep the layout healthy.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Far fewer files instead of hundreds (here just 1, because the whole demo table is smaller than the target file size), a much larger average file, and a faster query. On real data sizes, <code>OPTIMIZE</code> produces many files, and clustering gives each one a narrow <code>device_id</code> range, so a point filter reads only a few of them (lesson 03_10 measured that).
</div>

In [0]:
for apply in [lambda: spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.targetFileSize' = '2mb')"),
              lambda: spark.conf.set("spark.databricks.delta.optimize.maxFileSize", str(2 * 1024 * 1024))]:
    try:
        apply()          # small target size only so this small demo shows several files
    except Exception:
        pass

spark.sql(f"ALTER TABLE {table} CLUSTER BY (device_id)")
spark.sql(f"OPTIMIZE {table}").select("metrics.numFilesRemoved", "metrics.numFilesAdded").show()
layout_report(table, "device_id")
timed_query("after (clustered, compacted)")

## 5. Partitioning pitfalls, in numbers

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same data partitioned by a high-cardinality column, and reports the layout.<br><br>
<b>Why it matters</b><br>Partitioning by <code>device_id</code> (500 values) creates at least 500 folders and files even for tiny data. On big tables this is the most common cause of millions of small files.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Hundreds of files, each tiny, compared with the handful in the clustered table.
</div>

In [0]:
part_t = f"{DB}.readings_partitioned"
spark.table(table).write.mode("overwrite").partitionBy("device_id").saveAsTable(part_t)
layout_report(part_t, "reading")
spark.sql(f"DROP TABLE {part_t}")

## Under the hood

```
write with T tasks into P output partitions  ->  up to T x P files per write
N writes per day                              ->  N x T x P files per day
OPTIMIZE                                      ->  bin-pack into ~target size, cluster if CLUSTER BY is set
VACUUM (after retention)                      ->  physically delete the old small files
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> `OPTIMIZE` rewrites data (and is a commit). Compacting a huge table costs compute, so cluster and compact incrementally (liquid clustering does this) rather than rewriting everything repeatedly.

In [0]:
spark.sql(f"DESCRIBE HISTORY {table} LIMIT 4").select("version", "operation").show()
try:
    spark.conf.unset("spark.databricks.delta.optimize.maxFileSize")
except Exception:
    pass

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {table}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: thousands of tiny files after a few days of streaming</b><br>
No compaction. Enable optimized writes and auto compaction, or predictive optimization, and run <code>OPTIMIZE</code>.<br><br>
**⛔ Problem: <code>OPTIMIZE</code> runs every hour and costs a lot**<br>It rewrites more than needed. Use liquid clustering (incremental), and let predictive optimization schedule it.<br><br>
<b>⛔ Problem: storage keeps growing after compaction</b><br>The old files remain until <code>VACUUM</code>.<br><br>
<b>⛔ Problem: the table is partitioned by an ID</b><br>Rebuild without that partition column, and use liquid clustering on it.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. A job creates thousands of small files. Why, and what do you do?</b><br>
Each write task produces files, so many tasks, many output partitions or frequent micro-batches create many small files. I fix the source by reducing write parallelism with <code>coalesce</code>/<code>repartition</code>, enabling optimized writes and auto compaction, and avoiding high-cardinality partitioning. I fix existing tables with <code>OPTIMIZE</code> (with liquid clustering) and then <code>VACUUM</code>, and I let predictive optimization maintain them.<br><br>

<b>🎤 2. Why are small files bad?</b><br>
Every file costs listing, metadata, an open and a footer read, and many files mean more tasks and more planning time. With thousands of tiny files, this overhead dominates the actual reading, and the log and statistics grow too.<br><br>

<b>🎤 3. What is a good file size for Delta tables?</b><br>
Roughly tens of megabytes up to about 1 GB. <code>OPTIMIZE</code> targets about 1 GB by default, and frequently merged tables benefit from smaller files.<br><br>

<b>🎤 4. How do you measure a table's layout?</b><br>
<code>DESCRIBE DETAIL</code> for the file count and size, the <code>_metadata</code> column for per-file sizes and min/max of filter columns, and the query profile for files read and pruned.<br><br>

<b>🎤 5. What are optimized writes and auto compaction?</b><br>
Databricks table properties. Optimized writes shuffle data before writing so each partition gets fewer, larger files. Auto compaction runs a small <code>OPTIMIZE</code> after writes when it finds many small files.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A Delta table receives a micro-batch every minute and now has 200,000 files of about 100 KB. Which combination best fixes and prevents the problem?</b><br>
A. <code>VACUUM</code> only<br>
B. <code>OPTIMIZE</code> now, and enable auto compaction / optimized writes (or predictive optimization)<br>
C. Increase <code>spark.sql.shuffle.partitions</code><br>
D. Partition the table by <code>event_id</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which command reports the number of data files in a Delta table?</b><br>
A. <code>DESCRIBE HISTORY</code><br>
B. <code>DESCRIBE DETAIL</code><br>
C. <code>SHOW PARTITIONS</code><br>
D. <code>ANALYZE TABLE</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Recreate the bad-layout table with 30 micro-batches and run <code>layout_report</code></li><li>Fix it with <code>CLUSTER BY</code> and <code>OPTIMIZE</code>, and run <code>layout_report</code> again</li><li>Extend <code>layout_report</code> to also print the largest file size, and the number of distinct values of the column per file (on average)</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Bad layout again
spark.sql(f"CREATE OR REPLACE TABLE {table} (device_id INT, reading DOUBLE, event_ts TIMESTAMP)")
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'false', 'delta.autoOptimize.autoCompact' = 'false')")
for b in range(30):
    (spark.range(1000).repartition(4)
     .select((F.rand(seed=b) * 500).cast("int").alias("device_id"), F.rand().alias("reading"), F.current_timestamp().alias("event_ts"))
     .write.mode("append").saveAsTable(table))
layout_report(table, "device_id")

# 2. Cluster and compact
spark.sql(f"ALTER TABLE {table} CLUSTER BY (device_id)")
spark.sql(f"OPTIMIZE {table}")
layout_report(table, "device_id")

# 3. Extended report
def layout_report_v2(table, column):
    per_file = (spark.table(table).groupBy(F.col("_metadata.file_path").alias("f"))
                .agg(F.max("_metadata.file_size").alias("size"), F.countDistinct(column).alias("distinct_vals")))
    s = per_file.agg(F.max("size").alias("max_size"), F.avg("distinct_vals").alias("avg_distinct")).first()
    print(f"largest file: {s['max_size'] / 1024:.1f} KB | avg distinct {column} per file: {s['avg_distinct']:.0f}")

layout_report_v2(table, "device_id")
spark.sql(f"DROP TABLE {table}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Healthy files: tens of MB to ~1 GB, count proportional to data size, clustered on filter columns</li><li>Small files come from frequent appends, too many write tasks, over-partitioning and copy-on-write DML</li><li>Prevent: fewer write tasks, optimized writes, auto compaction, no high-cardinality partitioning</li><li>Repair: <code>OPTIMIZE</code> (with liquid clustering), then <code>VACUUM</code>. Predictive optimization automates both</li><li>Measure with a layout report: files, sizes, and per-file ranges of filter columns</li></ul>
</div>

| Task | Code |
|---|---|
| File count and size | `DESCRIBE DETAIL t` |
| Per-file info | `spark.table(t).select("_metadata.file_path", "_metadata.file_size")` |
| Fewer files on write | `df.coalesce(n)`, `delta.autoOptimize.optimizeWrite` |
| Compact | `OPTIMIZE t` |
| Cluster | `ALTER TABLE t CLUSTER BY (c)` + `OPTIMIZE t` |

## Git commit

```
git add 05_optimization/03_small_files_and_file_layout.ipynb
git commit -m "add 05_03 small files and file layout notebook"
```